In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
export OMP_NUM_THREADS=2
export OPENBLAS_NUM_THREADS=2
REF="43f62f3aff682b557bc00bbb4707479abbe268b7"
PREREG="fda9e069643e084d715c6f71a0cdefc9000f0204"
SHARD="0"
REPO=/kaggle/working/v12_mc3_repository
OUT="/kaggle/working/outputs/v12_mc3_cal/shard${SHARD}"
LOG="/kaggle/working/v12_mc3_cal_shard${SHARD}.log"
finish() {
  rc=$?
  trap - EXIT
  set +e
  if [ -d "$OUT" ]; then
    cp "$LOG" "$OUT/run.log"
    cd /kaggle/working
    tar -czf "v12_mc3_cal_shard${SHARD}.tgz" "outputs/v12_mc3_cal/shard${SHARD}"
  fi
  echo "[exit] rc=$rc shard=$SHARD"
  exit "$rc"
}
trap finish EXIT
git clone -q https://github.com/munnn01/preeee.git "$REPO"
git -C "$REPO" checkout -q "$REF"
test "$(git -C "$REPO" rev-parse HEAD)" = "$REF"
cd "$REPO"
python -m research.v12_mc3_audit.study --prereg-commit "$PREREG" preflight
ROOT=""
for candidate in /kaggle/input/kineticscleaned /kaggle/input/datasets/qktttttttttt/kineticscleaned; do
  if [ -d "$candidate" ]; then ROOT="$candidate"; break; fi
done
test -n "$ROOT" || { echo 'Locked CAL dataset missing' >&2; exit 2; }
python -m research.v12_mc3_audit.study --prereg-commit "$PREREG" shard --source-root "$ROOT" --shard "$SHARD" --out-dir "$OUT" 2>&1 | tee "$LOG"
test -s "$OUT/manifest.json"
test -s "$OUT/shard_records.jsonl"
echo "[done] V12 MC3 CAL shard=$SHARD"
